# Metodyka projektu: reprezentacja danych w analizie sygnału EKG

**Projekt:** Rozpoznawanie nieprawidłowych uderzeń serca na podstawie sygnału EKG  
**Przedmiot:** Akwizycja danych biomedycznych    
**Autorzy:** Ewa Kędziera, Jan Rapacz

---

Główny zarys projektu opiera się na analizie zapisu EKG na dwóch poziomach reprezentacji danych.

W projekcie wyróżniono dwie uzupełniające się części:

1. **Część I - reprezentacja rytmu**: analiza odstępów RR.
2. **Część II - reprezentacja sygnałowa**: klasyfikacja na podstawie cech sygnału oraz okien czasowych EKG.

Obie części dotyczą tego samego problemu, czyli klasyfikacji adnotowanych uderzeń serca z bazy MIT-BIH Arrhythmia Database. Różnica polega na tym, jaki rodzaj informacji jest przekazywany do algorytmu klasyfikującego. W pierwszej części wykorzystywana jest przede wszystkim informacja o regularności rytmu serca, natomiast w drugiej części uwzględniany jest również kształt sygnału EKG w otoczeniu danego uderzenia.

## 1. Jednostka analizy i cel klasyfikacji

Jednostką analizy jest **pojedyncze uderzenie serca** wskazane przez adnotację w bazie MIT-BIH. Dla każdego uderzenia dostępne są:

- identyfikator rekordu,
- pozycja próbki w sygnale,
- czas wystąpienia uderzenia,
- symbol adnotacji MIT-BIH,
- klasa AAMI,
- etykieta binarna: `normal` lub `abnormal`.

Główne zadanie ma postać klasyfikacji binarnej:

```text
uderzenie serca → normal / abnormal
```

## 2. Reprezentacje danych w analizie sygnału EKG

Sygnał EKG zawiera różne typy informacji. Najważniejsze dla projektu są dwa z nich:

| Typ informacji | Co opisuje? | Przykład reprezentacji |
|---|---|---|
| **rytm** | kiedy występują kolejne uderzenia | odstępy RR |
| **morfologia** | jak wygląda uderzenie | okno sygnału wokół piku R, cechy amplitudy, energii, szerokości QRS |

Podział projektu wynika z pytania badawczego:

> Czy do wykrywania nieprawidłowych uderzeń wystarczy sama informacja o rytmie, czy potrzebna jest także informacja o kształcie sygnału?

Dlatego najpierw budowany jest prosty baseline RR, a następnie metody wykorzystujące bogatszą reprezentację sygnału.

## 3. Wspólna baza przetwarzania

Obie części korzystają z tych samych założeń przygotowania danych:

```text
MIT-BIH Arrhythmia Database
→ wczytanie sygnałów i adnotacji
→ mapowanie symboli MIT-BIH do klas AAMI
→ definicja etykiety normal / abnormal
→ kontrola rekordów z paced beats
→ podział rekordów DS1 / DS2
→ ewaluacja bez przecieku między pacjentami
```

Najważniejsza zasada metodologiczna:

**Podział trening/test powinien być wykonywany po rekordach, a nie losowo po pojedynczych uderzeniach.**

Losowy podział uderzeń mógłby spowodować, że bardzo podobne próbki z tego samego pacjenta znalazłyby się jednocześnie w treningu i teście, co zawyżyłoby ocenę skuteczności.

## 4. Część I - klasyczna analiza odstępów RR

### Reprezentacja danych

W pierwszej części uderzenie jest opisane wyłącznie przez informację o czasie wystąpienia względem sąsiednich uderzeń.

Przykładowe cechy:

- `rr_prev_sec` - czas od poprzedniego uderzenia,
- `rr_next_sec` - czas do następnego uderzenia,
- lokalna mediana RR,
- `rr_ratio` - stosunek aktualnego RR do lokalnego rytmu,
- informacja, czy uderzenie wygląda na przedwczesne lub związane z pauzą.

### Idea

Wiele zaburzeń rytmu objawia się tym, że uderzenie pojawia się za wcześnie albo za późno względem lokalnego rytmu. Dlatego można zbudować prostą, interpretowalną regułę:

```text
jeśli RR jest wyraźnie krótszy niż lokalny rytm → potencjalne uderzenie przedwczesne
jeśli RR jest wyraźnie dłuższy niż lokalny rytm → potencjalna pauza / nieregularność
w przeciwnym razie → normal
```

### Rola w projekcie

Ta część pełni funkcję **baseline'u**. Pokazuje, ile można osiągnąć bez analizy kształtu zespołu QRS i bez trenowania złożonego modelu.

### Ograniczenia

Analiza RR jest celowo ograniczona. Wykorzystuje tylko informację o rytmie, więc nie rozpoznaje dobrze nieprawidłowości widocznych głównie w morfologii sygnału.

Przykładowe ograniczenia:

- brak analizy amplitudy i szerokości QRS,
- słaba separacja uderzeń o podobnym rytmie, ale różnym kształcie,
- możliwość fałszywych alarmów przy naturalnej zmienności rytmu,
- brak modelowania dłuższych epizodów klinicznych jako całości.

## 5. Część II - klasyfikacja na podstawie bogatszej reprezentacji sygnału

Druga część projektu koncentruje się na klasyfikacji uderzeń serca z wykorzystaniem informacji zawartej bezpośrednio w przebiegu sygnału EKG. W odróżnieniu od analizy odstępów RR, uwzględniany jest tutaj nie tylko moment wystąpienia uderzenia, ale również kształt fragmentu sygnału w jego otoczeniu.

```text
adnotacja uderzenia
→ okno sygnału przed i po uderzeniu
→ opis morfologii / wejście dla modelu
→ klasyfikacja normal / abnormal
```

W tej części są realizowane dwa warianty:

1. **klasyczne uczenie maszynowe na cechach**,  
2. **deep learning na oknach sygnału**.

Oba warianty korzystają z tej samej bogatszej reprezentacji sygnału. Różnią się tym, czy cechy są projektowane ręcznie, czy uczone automatycznie przez model.

### Klasyczne uczenie maszynowe na cechach

W tym podejściu uderzenia są klasyfikowane ina podstawie cech liczonych z okien sygnału EKG.

#### Reprezentacja danych

Dane wejściowe mogą być reprezentowane przez przykładowe grupy cech:

- cechy rytmu RR,
- cechy amplitudowe,
- cechy morfologiczne zespołu QRS,
- cechy energii i pola pod sygnałem,
- cechy pochodnych,
- proste cechy częstotliwościowe,
- wskaźniki jakości sygnału.

Przygotowane dane mają postać tabelaryczną, gdzie każdy wiersz odpowiada jednemu uderzeniu serca, a kolejne kolumny zawierają wartości cech opisujących to uderzenie.

Taka reprezentacja może być następnie wykorzystana przez klasyczne algorytmy uczenia maszynowego.

#### Pipeline

```text
EKG
→ filtracja
→ piki R / adnotacje
→ okna wokół R
→ ekstrakcja cech
→ skalowanie
→ klasyfikator ML
→ ewaluacja
```

#### Przykładowe modele

- Logistic Regression,
- k-NN,
- SVM,
- Random Forest,
- Gradient Boosting,
- XGBoost/LightGBM, jeśli projekt ma być bardziej zaawansowany.

### Deep learning na oknach sygnału

W wariancie deep learning uderzenia serca są klasyfikowane bezpośrednio na podstawie okien sygnału EKG wyciętych wokół adnotowanych uderzeń. W odróżnieniu od klasycznego uczenia maszynowego nie trzeba ręcznie definiować wszystkich cech opisujących kształt sygnału. Model otrzymuje fragment przebiegu EKG i sam uczy się wzorców, które mogą odróżniać uderzenia prawidłowe od nieprawidłowych.

#### Reprezentacja danych

Dane wejściowe mają postać sekwencji próbek sygnału. Każde okno odpowiada jednemu uderzeniu serca i obejmuje fragment sygnału przed oraz po piku R.

Typowa reprezentacja wejścia może zostać zapisana jako:

```text
liczba_uderzeń × liczba_próbek × liczba_kanałów
```

Dla jednego kanału EKG i okna o długości 252 próbek otrzymujemy:

```text
N × 252 × 1
```

gdzie `N` oznacza liczbę analizowanych uderzeń serca, `252` oznacza liczbę próbek w pojedynczym oknie, a `1` oznacza jeden kanał sygnału EKG.

#### Pipeline

```text
EKG
→ filtracja
→ piki R / adnotacje
→ okna wokół R
→ normalizacja okien
→ model deep learning
→ klasyfikacja
→ ewaluacja
```

W tym wariancie etap ekstrakcji cech zostaje zastąpiony przez automatyczne uczenie reprezentacji przez model.

#### Model

Naturalnym wyborem dla tego zadania jest sieć 1D CNN, ponieważ konwolucje jednowymiarowe dobrze nadają się do analizy przebiegów czasowych. Mogą wykrywać lokalne wzorce w sygnale, takie jak gwałtowne zmiany amplitudy, kształt zespołu QRS, szerokość uderzenia lub różnice między fragmentami przed i po piku R.

#### Zalety

- model uczy się cech automatycznie,
- może uchwycić złożoną morfologię uderzeń,
- dobre rozszerzenie projektu ML.

#### Ograniczenia

- wymaga więcej danych,
- trudniejszy do interpretacji,
- łatwo o overfitting,
- wymaga ostrożnego podziału danych.

## 6. Metryki oceny

Problem klasyfikacji arytmii jest niezbalansowany. Uderzeń prawidłowych jest zwykle znacznie więcej niż nieprawidłowych, dlatego sama accuracy może być myląca.

W projekcie należy raportować przede wszystkim:

| Metryka | Znaczenie |
|---|---|
| **precision** | ile przewidzianych uderzeń nieprawidłowych rzeczywiście było nieprawidłowych |
| **recall / sensitivity** | ile rzeczywistych uderzeń nieprawidłowych zostało wykrytych |
| **specificity** | ile uderzeń prawidłowych poprawnie rozpoznano jako prawidłowe |
| **F1-score** | kompromis między precision i recall |
| **confusion matrix** | pełny rozkład TP, FP, TN i FN |
| **PR-AUC** | szczególnie przydatne przy niezbalansowanych klasach |

Dla wykrywania uderzeń nieprawidłowych szczególnie ważny jest recall klasy `abnormal`, ale nie można ignorować precision, ponieważ nadmiar fałszywych alarmów obniża praktyczną użyteczność metody.